In [1]:
import pandas as pd

data = pd.read_csv('APPLICATIONS_DATA.csv')
data.head()

,APPLICATION ID,NAME,EMAIL,SUBMISSION TIME,AGE,EXPERIENCE YEARS,TIME GAP SETTING
0,1,ALI KHAN,alikhan@gmail.com,48720,24,1,0
1,2,SARA AHMED,saraahmed@gmail.com,68810,20,3,20090
2,3,BILAL HUSSAIN,bilalhussain@gmail.com,6748,27,3,62062
3,4,AYESHA MALIK,ayeshamalik@gmail.com,39988,24,2,33240
4,5,USMAN TARIQ,usmantariq@gmail.com,68903,15,1,28915


In [2]:
data.columns = data.columns.str.strip()
print(data.columns.tolist())

['APPLICATION ID', 'NAME', 'EMAIL', 'SUBMISSION TIME', 'AGE', 'EXPERIENCE YEARS', 'TIME GAP SETTING']


In [3]:
features = data[['AGE', 'EXPERIENCE YEARS', 'TIME GAP SETTING']]

In [4]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(contamination=0.15, random_state=42)
data['Anomaly_IsolationForest'] = iso_model.fit_predict(features)

In [5]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
data['Cluster'] = kmeans.fit_predict(features)

In [6]:
duplicates = data[data.duplicated(subset=['NAME', 'EMAIL'], keep=False)]
print(duplicates)

    APPLICATION ID         NAME                 EMAIL  SUBMISSION TIME  AGE  \
9               10  FAHAD IQBAL  fahadiqbal@gmail.com            57688   28   
23              24  FAHAD IQBAL  fahadiqbal@gmail.com            74155   19   

    EXPERIENCE YEARS  TIME GAP SETTING  Anomaly_IsolationForest  Cluster  
9                  0             11966                        1        1  
23                 3             51798                        1        2  


In [7]:
data['Suspicious'] = data['Anomaly_IsolationForest'].apply(lambda x: 'Yes' if x == -1 else 'No')
suspicious_entries = data[data['Suspicious'] == 'Yes']
print(suspicious_entries)

    APPLICATION ID           NAME                   EMAIL  SUBMISSION TIME  \
2                3  BILAL HUSSAIN  bilalhussain@gmail.com             6748   
4                5    USMAN TARIQ    usmantariq@gmail.com            68903   
11              12    KASHIF RAZA    kashifraza@gmail.com            10330   
15              16    OMAR FAROOQ    omarfarooq@gmail.com            29604   
29              30    NIMRA ANWAR    nimraanwar@gmail.com            36064   
35              36  SOBIA PARVEEN  sobiaparveen@gmail.com             1251   

    AGE  EXPERIENCE YEARS  TIME GAP SETTING  Anomaly_IsolationForest  Cluster  \
2    27                 3             62062                       -1        2   
4    15                 1             28915                       -1        0   
11   22                20             21616                       -1        0   
15   28                 3                 5                       -1        1   
29   45                 0                23     

In [8]:
print("Total flagged:", len(suspicious_entries))
print("Percentage flagged:", len(suspicious_entries)/len(data)*100, "%")

Total flagged: 6
Percentage flagged: 15.0 %


In [9]:
print(data.groupby('Cluster')[['AGE','EXPERIENCE YEARS','TIME GAP SETTING']].mean())
print(data['Cluster'].value_counts())

               AGE  EXPERIENCE YEARS  TIME GAP SETTING
Cluster                                               
0        21.764706          2.588235      26629.117647
1        24.923077          1.384615       5129.461538
2        22.500000          1.500000      47922.300000
Cluster
0    17
1    13
2    10
Name: count, dtype: int64


In [10]:
for index, row in suspicious_entries.iterrows():
    print(f"⚠️ ALERT: Application ID {row['APPLICATION ID']} ({row['NAME']}) flagged as suspicious.")

⚠️ ALERT: Application ID 3 (BILAL HUSSAIN) flagged as suspicious.
⚠️ ALERT: Application ID 5 (USMAN TARIQ) flagged as suspicious.
⚠️ ALERT: Application ID 12 (KASHIF RAZA) flagged as suspicious.
⚠️ ALERT: Application ID 16 (OMAR FAROOQ) flagged as suspicious.
⚠️ ALERT: Application ID 30 (NIMRA ANWAR) flagged as suspicious.
⚠️ ALERT: Application ID 36 (SOBIA PARVEEN) flagged as suspicious.


The anomaly detection system analyzed 40 internship applications using Isolation Forest, K-Means clustering, and duplicate-entry detection.

Isolation Forest flagged 6 applications (15%) as anomalies based on unusual age, experience, or submission-timing patterns. Separately, duplicate-entry detection identified 2 applications submitted under the same name and email — a case Isolation Forest could not catch on its own, since it does not evaluate identity fields.

K-Means clustering grouped applicants into 3 behavioral segments, primarily differentiated by submission speed: one cluster showed a notably faster average submission gap (5,100 seconds) compared to the other two (27,000 and 48,000 seconds), helping to contextualize which applicants submitted unusually quickly.

An automated alert system was implemented to flag suspicious applications individually by ID and name, allowing reviewers to act on specific cases rather than manually scanning the full dataset.

Recommendation: Combining statistical outlier detection (Isolation Forest), behavioral clustering (K-Means), and rule-based identity checks (duplicates) provides more reliable fraud detection than any single method. Flagged applications should be routed for human review rather than automatically rejected, to avoid penalizing legitimate edge cases.
